# 02 — SPY vol-target backtest

**Objective:** target 10% annualised portfolio vol on SPY. Position size =
`target_vol / forecast_vol`, clipped to [0, max_leverage]. Forecast = trailing
21-day realised vol.

**Bias controls:**
- All inputs at time t use only data with ts ≤ t.
- Rebalance at next-bar OPEN (engine shifts the weight by 1 bar).
- Pre-flight leakage check runs automatically.

**Failure conditions:** if pre-flight detects leakage, the run aborts with an error.

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
from src.data.yfinance_client import YFinanceClient
from src.strategies.spy_vol_target import SPYVolTargetStrategy, VolTargetConfig
from src.backtest.engine import run_backtest, EngineConfig
from src.reports.tearsheet import tearsheet

yf = YFinanceClient()
spy = yf.get_daily_bars('SPY', start='2010-01-01')
prices = pd.DataFrame({'SPY': spy['adj_close']})
print('SPY rows:', len(prices))

### Configure

In [ ]:
cfg_strat = VolTargetConfig(symbol='SPY', target_vol=0.10, rv_window=21, max_leverage=1.0)
cfg_engine = EngineConfig(starting_cash=100_000, rebalance_freq='D',
                          benchmark='SPY', strategy_name='spy_vol_target')
strat = SPYVolTargetStrategy(cfg_strat)
result = run_backtest(strat, prices, cfg_engine)
result['metrics']

### Tearsheet

In [ ]:
fig, m = tearsheet(result, title='SPY vol-target')
plt.show()

### Interpretation

Vol-targeting is a risk-control overlay, not an alpha strategy. The expected outcome:
- Realised vol close to 10% target.
- Sharpe similar to or modestly better than buy-and-hold SPY.
- **Materially smaller drawdowns** — that's the whole point.

If you see Sharpe much higher than buy-and-hold's, double-check for lookahead.

### Sweep target vols

In [ ]:
rows = []
for tv in [0.05, 0.08, 0.10, 0.15, 0.20]:
    s = SPYVolTargetStrategy(VolTargetConfig(symbol='SPY', target_vol=tv,
                                              rv_window=21, max_leverage=2.0))
    r = run_backtest(s, prices, EngineConfig(starting_cash=100_000, rebalance_freq='D',
                                              benchmark='SPY', strategy_name=f'svt_{tv}',
                                              run_preflight=False))
    rows.append(dict(target_vol=tv, **{k: r['metrics'][k] for k in
                       ['sharpe','cagr','ann_vol','max_drawdown','calmar']}))
pd.DataFrame(rows)

### Next steps

1. Replace 21-day realised vol with EWMA / GARCH forecast.
2. Add ML vol forecast (only with walk-forward training; baseline must work first).
3. Run `07_live_paper_trading_dashboard.ipynb` to paper trade this.
4. Combine with TSMOM in `06_combined_portfolio_backtest.ipynb`.